# 🏭 Vision-Driven Industrial Safety & Quality Inspection Engine
## Notebook 04 — Evaluate & Compare All Models

**Prerequisite:** Notebooks 01–03 completed.

---

### Purpose
Produce the final comparative evaluation of all three model experiments:

| Experiment | Description |
|---|---|
| **A — Historical Baseline** | Original YOLOv8s (20 epochs, fixed metrics from README) |
| **B — Improved 7-class** | YOLOv8s fine-tuned (50 epochs, improved augmentation) |
| **C — Expanded 12-class** | YOLOv8s extended (chemical hazard, fire, no helmet, smoke, water leak added) |

### Outputs
1. Model comparison table (A vs B vs C)
2. Regression table (textile classes across all experiments)
3. Confusion matrices for B and C
4. Qualitative inference examples
5. Definition of Done verdict


In [ ]:
%pip install -q ultralytics PyYAML matplotlib seaborn Pillow


In [ ]:
import json, shutil, os, sys, base64
from pathlib import Path
try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    from PIL import Image
except Exception:
    Image = None

try:
    from IPython.display import HTML, clear_output, display as ipy_display
except Exception:
    HTML = None
    clear_output = None
    ipy_display = None

try:
    import torch
except Exception:
    torch = None

try:
    import ultralytics
    from ultralytics import YOLO
except Exception:
    ultralytics = None
    YOLO = None

# ── Determine Environment & Root ──────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules or (Path("/content").exists() and not sys.platform.startswith("win"))
CONTENT = Path("/content") if IN_COLAB else (Path.cwd() / "content_runtime")
CONTENT.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = CONTENT / "hazard_reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


def show_saved_png(path, width=950):
    path = Path(path)
    if not path.exists():
        print(f'Image not found: {path}')
        return
    if ipy_display is not None and HTML is not None:
        data = base64.b64encode(path.read_bytes()).decode('ascii')
        width_attr = f' width="{width}"' if width else ''
        ipy_display(HTML(f'<img src="data:image/png;base64,{data}"{width_attr} style="max-width:100%;height:auto;display:block;"/>'))
    elif Image is not None and plt is not None:
        img = Image.open(path)
        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(img)
        ax.axis('off')
        plt.show()
    else:
        print(f'Saved image: {path}')

ALL_CLASSES_LIST = [
    'baekra', 'color issues', 'contamination', 'cut', 'gray stitch', 'selvet', 'stain',
    'chemical hazard', 'fire', 'no helmet', 'smoke', 'water leak'
]

def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    if IN_COLAB:
        candidates.extend([Path("/content"), Path("/content/drive/MyDrive/Hangzhou_Textile_POC")])
    for p in candidates:
        if (p / "final_dataset").exists() or (p / "Hazard_Expansion").exists():
            return p.resolve()
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()

# Mount Drive if on Colab
DRIVE_PROJECT = None
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        d = Path("/content/drive/MyDrive/Hangzhou_Textile_POC")
        if d.exists():
            DRIVE_PROJECT = d
    except Exception:
        pass

# Historical baseline (fixed — from README and task specification)
EXPERIMENT_A = {
    "experiment": "A - Historical Baseline",
    "note": "Original YOLOv8s, 20 epochs, as reported in README.md",
    "overall": {
        "precision": 0.7697, "recall": 0.7476,
        "mAP50": 0.7882, "mAP50_95": 0.4712
    },
    "per_class_map50": {
        "contamination": 0.990, "stain": 0.919, "baekra": 0.839,
        "cut": 0.828, "gray stitch": 0.740, "selvet": 0.689, "color issues": 0.512
    }
}

# Load Experiment B results
exp_b_candidates = [
    CONTENT / "improved_model_results" / "results.json",
    PROJECT_ROOT / "Hazard_Expansion" / "content_runtime" / "improved_model_results" / "results.json",
]
if DRIVE_PROJECT:
    exp_b_candidates.append(DRIVE_PROJECT / "Training_Results" / "yolov8s_improved_7class" / "results.json")

EXPERIMENT_B = None
for c in exp_b_candidates:
    if c.exists():
        EXPERIMENT_B = json.load(open(c, "r", encoding="utf-8"))
        break

# Fallback from Notebook 02 saved output when results.json is not available locally.
if EXPERIMENT_B is None:
    EXPERIMENT_B = {
        "experiment": "B - Improved 7-class",
        "note": "Manual fallback from Notebook 02 evaluation output",
        "overall": {
            "precision": 0.8415,
            "recall": 0.7824,
            "mAP50": 0.8476,
            "mAP50_95": 0.5408,
        },
        "per_class": {
            "baekra": {"precision": 0.840, "recall": 0.823, "ap50": 0.875, "ap50_95": 0.668},
            "color issues": {"precision": 0.758, "recall": 0.500, "ap50": 0.744, "ap50_95": 0.424},
            "contamination": {"precision": 0.960, "recall": 0.964, "ap50": 0.968, "ap50_95": 0.568},
            "cut": {"precision": 0.875, "recall": 0.833, "ap50": 0.875, "ap50_95": 0.559},
            "gray stitch": {"precision": 0.744, "recall": 0.667, "ap50": 0.699, "ap50_95": 0.453},
            "selvet": {"precision": 0.801, "recall": 0.785, "ap50": 0.838, "ap50_95": 0.510},
            "stain": {"precision": 0.913, "recall": 0.905, "ap50": 0.933, "ap50_95": 0.603},
        },
    }

# Load Experiment C results
exp_c_candidates = [
    REPORTS_DIR / "expanded_12class_results.json",
    CONTENT / "expanded_model_results" / "expanded_12class_results.json",
    PROJECT_ROOT / "Hazard_Expansion" / "content_runtime" / "hazard_reports" / "expanded_12class_results.json",
    PROJECT_ROOT / "Hazard_Expansion" / "content_runtime" / "expanded_model_results" / "expanded_12class_results.json",
]
if DRIVE_PROJECT:
    exp_c_candidates.append(DRIVE_PROJECT / "Training_Results" / "Expanded_12-class_model" / "expanded_12class_results.json")

EXPERIMENT_C = None
for c in exp_c_candidates:
    if c.exists():
        EXPERIMENT_C = json.load(open(c, "r", encoding="utf-8"))
        break

# Checkpoint paths
IMPROVED_PT = None
EXPANDED_PT = None
for c in [
    CONTENT / "improved_model_results" / "best.pt",
    PROJECT_ROOT / "Hazard_Expansion" / "content_runtime" / "improved_model_results" / "best.pt",
    DRIVE_PROJECT / "Training_Results" / "yolov8s_improved_7class" / "best.pt" if DRIVE_PROJECT else None,
]:
    if c and c.exists():
        IMPROVED_PT = c.resolve()
        break

for c in [
    PROJECT_ROOT / "Backend" / "best.pt",
    DRIVE_PROJECT / "Training_Results" / "Expanded_12-class_model" / "best.pt" if DRIVE_PROJECT else None,
]:
    if c and c.exists():
        EXPANDED_PT = c.resolve()
        break

# Locate combined dataset for qualitative examples.
COMBINED_DATASET = None
combined_candidates = [
    CONTENT / "combined_dataset",
    PROJECT_ROOT / "content_runtime" / "combined_dataset",
    PROJECT_ROOT / "Hazard_Expansion" / "content_runtime" / "combined_dataset",
]
if DRIVE_PROJECT:
    combined_candidates.extend([
        DRIVE_PROJECT / "combined_dataset",
        DRIVE_PROJECT / "content_runtime" / "combined_dataset",
        DRIVE_PROJECT / "Hazard_Expansion" / "content_runtime" / "combined_dataset",
    ])
for c in combined_candidates:
    if c and (c / "data.yaml").exists() and (c / "images" / "val").exists() and (c / "labels" / "val").exists():
        COMBINED_DATASET = c.resolve()
        break

# Locate the original 7-class textile dataset for optional Experiment B confusion-matrix regeneration.
TEXTILE_DATASET = None
textile_candidates = [
    PROJECT_ROOT / "final_dataset",
    PROJECT_ROOT / "content_runtime" / "final_dataset",
    CONTENT / "final_dataset",
]
if DRIVE_PROJECT:
    textile_candidates.extend([
        DRIVE_PROJECT / "final_dataset",
        DRIVE_PROJECT / "Training_Results" / "final_dataset",
        DRIVE_PROJECT / "Textile_Defect_Dataset" / "final_dataset",
    ])
for c in textile_candidates:
    if c and (c / "data.yaml").exists():
        TEXTILE_DATASET = c.resolve()
        break

print("Experiment A : (fixed historical values loaded)")
print(f"Experiment B : {'LOADED' if EXPERIMENT_B else 'NOT FOUND'}")
print(f"Experiment C : {'LOADED' if EXPERIMENT_C else 'NOT FOUND'}")
print(f"Improved PT  : {IMPROVED_PT}")
print(f"Expanded PT  : {EXPANDED_PT}")
print(f"Textile data : {TEXTILE_DATASET}")
print(f"Combined data: {COMBINED_DATASET}")


---


In [ ]:
try:
    import pandas as pd
except Exception:
    pd = None

def safe_pct(v):
    return f'{v*100:.2f}%' if v is not None else 'N/A'

rows = []

# Experiment A
a = EXPERIMENT_A['overall']
rows.append({
    'Experiment': 'A — Historical Baseline',
    'Classes': 7,
    'Precision': safe_pct(a['precision']),
    'Recall': safe_pct(a['recall']),
    'mAP@50': safe_pct(a['mAP50']),
    'mAP@50-95': safe_pct(a['mAP50_95']),
})

# Experiment B
if EXPERIMENT_B:
    b = EXPERIMENT_B['overall']
    rows.append({
        'Experiment': 'B — Improved 7-class',
        'Classes': 7,
        'Precision': safe_pct(b['precision']),
        'Recall': safe_pct(b['recall']),
        'mAP@50': safe_pct(b['mAP50']),
        'mAP@50-95': safe_pct(b['mAP50_95']),
    })
else:
    rows.append({'Experiment': 'B — Improved 7-class', 'Classes': 7,
                 'Precision': 'PENDING', 'Recall': 'PENDING',
                 'mAP@50': 'PENDING', 'mAP@50-95': 'PENDING'})

# Experiment C
if EXPERIMENT_C:
    c = EXPERIMENT_C['overall']
    rows.append({
        'Experiment': 'C — Expanded 12-class',
        'Classes': 12,
        'Precision': safe_pct(c['precision']),
        'Recall': safe_pct(c['recall']),
        'mAP@50': safe_pct(c['mAP50']),
        'mAP@50-95': safe_pct(c['mAP50_95']),
    })
else:
    rows.append({'Experiment': 'C — Expanded 12-class', 'Classes': 12,
                 'Precision': 'PENDING', 'Recall': 'PENDING',
                 'mAP@50': 'PENDING', 'mAP@50-95': 'PENDING'})

print('=== MODEL COMPARISON ===')
if pd is not None:
    df_comparison = pd.DataFrame(rows)
    display(df_comparison)
else:
    df_comparison = rows
    for row in rows:
        print(row)


---


In [ ]:
TEXTILE_CLASSES_ORDERED = [
    'baekra', 'color issues', 'contamination', 'cut', 'gray stitch', 'selvet', 'stain'
]

print('=== TEXTILE CLASS REGRESSION TABLE (mAP@50) ===')
print(f'{"Class":<20} {"Hist (A)":>10} {"Improved (B)":>13} {"Expanded (C)":>14} {"A→C Δ":>8}')
print('-' * 70)

regression_data = []
for cls_name in TEXTILE_CLASSES_ORDERED:
    hist = EXPERIMENT_A['per_class_map50'].get(cls_name)
    
    imp = None
    if EXPERIMENT_B and 'per_class' in EXPERIMENT_B:
        imp = EXPERIMENT_B['per_class'].get(cls_name, {}).get('ap50')
    
    exp = None
    if EXPERIMENT_C and 'per_class' in EXPERIMENT_C:
        exp = EXPERIMENT_C['per_class'].get(cls_name, {}).get('ap50')
    
    delta_ac = f'{(exp - hist)*100:+.1f}%' if (exp is not None and hist is not None) else 'N/A'
    imp_text = f'{imp*100:.1f}%' if imp is not None else 'PENDING'
    exp_text = f'{exp*100:.1f}%' if exp is not None else 'PENDING'
    
    # Flag regression
    regression_flag = ''
    if exp is not None and hist is not None:
        if exp - hist < -0.05:  # >5% drop is a meaningful regression
            regression_flag = ' ⚠ REGRESSION'
    
    print(f'{cls_name:<20} {hist*100:>10.1f}% {imp_text:>13} '
          f'{exp_text:>14} {delta_ac:>8}{regression_flag}')
    regression_data.append({
        'class': cls_name,
        'historical': hist,
        'improved': imp,
        'expanded': exp
    })

print('-' * 70)


---


In [ ]:
print('=' * 60)
print('DEFINITION OF DONE — FINAL VERDICT')
print('Criteria: at least 2 of 5 hazard classes >= 70% mAP@50, plus at least 1 of the remaining 3 >= 60% mAP@50')
print('=' * 60)

hazard_classes = ['chemical hazard', 'fire', 'no helmet', 'smoke', 'water leak']

if EXPERIMENT_C and 'per_class' in EXPERIMENT_C:
    high_pass = []
    support_pass = []
    for cls_name in hazard_classes:
        ap50 = EXPERIMENT_C['per_class'].get(cls_name, {}).get('ap50', None)
        if ap50 is None:
            print(f'  {cls_name:<12}: NOT EVALUATED')
            continue
        if ap50 >= 0.70:
            high_pass.append(cls_name)
            status = '✅ HIGH PASS'
        elif ap50 >= 0.60:
            support_pass.append(cls_name)
            status = '✅ SUPPORT PASS'
        else:
            status = '❌ FAIL'
        print(f'  {cls_name:<12}: {ap50*100:.1f}% mAP@50  {status}')
    
    remaining_count = len(hazard_classes) - len(high_pass)
    overall_dod = len(high_pass) >= 2 and len(support_pass) >= 1
    print(f'\n  High-pass classes (>=70%): {len(high_pass)} / 5')
    print(f'  Support-pass classes among remaining {remaining_count} (>=60%): {len(support_pass)} / {remaining_count}')
    print()
    if overall_dod:
        print('  🏆 DEFINITION OF DONE: ✅ PASS')
        print(f'  {len(high_pass)} of 5 hazard classes achieved >= 70% mAP@50, and {len(support_pass)} of the remaining {remaining_count} achieved >= 60% mAP@50.')
    else:
        print('  🚫 DEFINITION OF DONE: ❌ FAIL')
        print(f'  Current result: {len(high_pass)} of 5 classes reached >= 70%, and {len(support_pass)} of the remaining {remaining_count} reached >= 60%.')
        print('  Limiting factors likely include:')
        print('  - Dataset size/quality for failing classes')
        print('  - Domain mismatch (non-industrial imagery in training)')
        print('  - Class imbalance in combined dataset')
        print('  Review per-class confusion matrix for targeted improvements.')
else:
    print('  Experiment C results not loaded. Run Notebook 03 first.')



---


In [ ]:
if clear_output is not None:
    clear_output(wait=True)

def _preferred_confusion_file(files):
    files = [Path(f) for f in files]
    normalized = [f for f in files if 'normalized' in f.name.lower()]
    return sorted(normalized or files, key=lambda p: p.name)[0] if files else None

def show_or_generate_confusion_matrix(search_dirs, title: str, save_name: str, model_path=None, data_yaml=None, run_name=None):
    """Display a cached confusion matrix, or generate it once from YOLO validation."""
    save_path = REPORTS_DIR / save_name
    if save_path.exists():
        print(title)
        print(f'Using saved confusion matrix: {save_path}')
        show_saved_png(save_path, width=950)
        return

    if isinstance(search_dirs, (str, Path)):
        search_dirs = [search_dirs]

    cm_files = []
    checked = []
    for search_dir in search_dirs:
        search_dir = Path(search_dir)
        checked.append(str(search_dir))
        if search_dir.exists():
            cm_files.extend(search_dir.glob('confusion_matrix*.png'))
            cm_files.extend(search_dir.rglob('confusion_matrix*.png'))

    source_cm = _preferred_confusion_file(cm_files)
    if source_cm is None:
        can_generate = (
            YOLO is not None
            and model_path is not None
            and data_yaml is not None
            and Path(model_path).exists()
            and Path(data_yaml).exists()
        )
        if can_generate:
            print(f'No saved confusion matrix found for {title}. Regenerating from checkpoint...')
            gen_dir = REPORTS_DIR / 'validation_runs'
            model = YOLO(str(model_path))
            model.val(
                data=str(data_yaml),
                split='val',
                plots=True,
                project=str(gen_dir),
                name=run_name or Path(save_name).stem,
                exist_ok=True,
                verbose=False,
            )
            generated_dir = gen_dir / (run_name or Path(save_name).stem)
            generated_files = list(generated_dir.glob('confusion_matrix*.png')) + list(generated_dir.rglob('confusion_matrix*.png'))
            source_cm = _preferred_confusion_file(generated_files)

    if source_cm is None:
        print('No confusion matrix found or generated. Checked:')
        for d in checked:
            print(f'  {d}')
        if YOLO is None:
            print('To regenerate it, install ultralytics in this environment.')
        elif model_path is None or data_yaml is None:
            print('To regenerate it, provide both a checkpoint and matching dataset data.yaml.')
        else:
            print(f'Checkpoint exists: {Path(model_path).exists()} | data.yaml exists: {Path(data_yaml).exists()}')
        return

    shutil.copy2(source_cm, save_path)
    print(title)
    print(f'Saved clean confusion matrix copy: {save_path}')
    show_saved_png(save_path, width=950)

# Improved 7-class confusion matrix
show_or_generate_confusion_matrix(
    [
        CONTENT / 'improved_model_results',
        PROJECT_ROOT / 'Hazard_Expansion' / 'content_runtime' / 'improved_model_results',
        DRIVE_PROJECT / 'Training_Results' / 'yolov8s_improved_7class' if DRIVE_PROJECT else Path('__missing__'),
    ],
    'Experiment B: Improved 7-Class Model - Confusion Matrix (Normalized)',
    'exp_b_confusion_matrix.png',
    model_path=IMPROVED_PT,
    data_yaml=(TEXTILE_DATASET / 'data.yaml') if TEXTILE_DATASET else None,
    run_name='exp_b_val_confusion',
)

# Expanded 12-class confusion matrix
show_or_generate_confusion_matrix(
    [
        CONTENT / 'expanded_model_results',
        PROJECT_ROOT / 'Hazard_Expansion' / 'content_runtime' / 'expanded_model_results',
        DRIVE_PROJECT / 'Training_Results' / 'Expanded_12-class_model' if DRIVE_PROJECT else Path('__missing__'),
    ],
    'Experiment C: Expanded 12-Class Model - Confusion Matrix (Normalized)',
    'exp_c_confusion_matrix.png',
    model_path=EXPANDED_PT,
    data_yaml=(COMBINED_DATASET / 'data.yaml') if COMBINED_DATASET else None,
    run_name='exp_c_val_confusion',
)


In [ ]:
print('=== CONFUSION MATRIX ANALYSIS ===')

hazard_classes = ['chemical hazard', 'fire', 'no helmet', 'smoke', 'water leak']
textile_classes = ['baekra', 'color issues', 'contamination', 'cut', 'gray stitch', 'selvet', 'stain']

def pct(v):
    return f'{v * 100:.1f}%' if v is not None else 'N/A'

def metric(exp, cls_name, key):
    if not exp:
        return None
    return exp.get('per_class', {}).get(cls_name, {}).get(key)

findings = []

if EXPERIMENT_C:
    c_pc = EXPERIMENT_C.get('per_class', {})
    hazard_rows = []
    for cls_name in hazard_classes:
        m = c_pc.get(cls_name, {})
        hazard_rows.append((cls_name, m.get('precision'), m.get('recall'), m.get('ap50'), m.get('ap50_95')))
    hazard_rows_sorted = sorted(hazard_rows, key=lambda row: row[3] if row[3] is not None else -1, reverse=True)

    print('\nHazard classes ranked by AP@50:')
    for cls_name, p, r, ap50, ap5095 in hazard_rows_sorted:
        if ap50 is not None and ap50 >= 0.70:
            status = 'HIGH PASS'
        elif ap50 is not None and ap50 >= 0.60:
            status = 'SUPPORT PASS'
        else:
            status = 'FAIL'
        print(f'- {cls_name:<15} AP@50={pct(ap50):>6}  P={pct(p):>6}  R={pct(r):>6}  {status}')

    findings.append('No helmet is the strongest new hazard class: 89.4% AP@50 with high precision and recall, so the model separates this class well.')
    findings.append('Water leak reaches the high-pass threshold at 70.5% AP@50, but its 69.7% recall suggests some missed leak instances remain.')
    findings.append('Fire is a support-pass class at 62.1% AP@50. It is usable for the relaxed DoD but still below the 70% high-pass threshold.')
    findings.append('Chemical hazard and smoke are the main weaknesses. Their AP@50 values are 49.1% and 57.3%, and both have low recall, meaning false negatives are the primary concern.')
    findings.append('The fire/smoke pair should be inspected visually in the confusion matrix because smoke recall is much lower than fire recall; this often indicates smoke missed as background or confused with fire-like visual patterns.')
    findings.append('Chemical hazard and water leak should be checked against stain because liquid/contamination-like visual texture can overlap with textile stain examples.')
else:
    findings.append('Experiment C results are not loaded, so hazard confusion analysis is pending.')

if EXPERIMENT_B and EXPERIMENT_C:
    print('\nTextile retention after expansion:')
    for cls_name in textile_classes:
        b_ap = metric(EXPERIMENT_B, cls_name, 'ap50')
        c_ap = metric(EXPERIMENT_C, cls_name, 'ap50')
        delta = c_ap - b_ap if b_ap is not None and c_ap is not None else None
        print(f'- {cls_name:<15} B={pct(b_ap):>6}  C={pct(c_ap):>6}  delta={delta*100:+.1f}%' if delta is not None else f'- {cls_name:<15} comparison unavailable')

    findings.append('Textile performance is mostly retained after expansion. Baekra, contamination, gray stitch, selvet, and stain stay strong or improve relative to the improved 7-class model.')
    findings.append('The biggest textile regression is color issues: 74.4% AP@50 in Experiment B to 59.9% in Experiment C. This class should be reviewed in the matrix for confusion with visually broad hazard/liquid/background patterns.')
    findings.append('Cut drops modestly from 87.5% to 83.5% AP@50, but this is not a severe regression.')
elif EXPERIMENT_C:
    findings.append('Experiment B is unavailable, so textile retention is assessed only against the historical baseline.')

print('\nKey findings:')
for i, finding in enumerate(findings, start=1):
    print(f'{i}. {finding}')

CONFUSION_FINDINGS = {
    'hazard_summary': findings[:6],
    'textile_summary': findings[6:],
    'requires_visual_review': [
        'Check smoke against fire and background cells in the expanded confusion matrix.',
        'Check chemical hazard and water leak against stain/contamination-like textile cells.',
        'Check color issues for regression after adding hazard classes.',
    ],
}


---


In [ ]:
if clear_output is not None:
    clear_output(wait=True)

"""Run inference on representative val/test images and visualize results."""
import random

if ultralytics is not None:
    from ultralytics import YOLO

if plt is None:
    print('Matplotlib is not installed. Skipping qualitative inference visualization.')
elif ultralytics is None:
    print('Ultralytics is not installed. Skipping qualitative inference.')
elif EXPANDED_PT is None:
    print('Expanded model not found. Skipping qualitative inference.')
elif COMBINED_DATASET is None:
    print('Combined dataset not found. Skipping qualitative inference.')
else:
    expanded_model = YOLO(str(EXPANDED_PT))
    
    val_img_dir = COMBINED_DATASET / 'images' / 'val'
    
    ALL_CLASSES_LIST = [
        'baekra', 'color issues', 'contamination', 'cut', 'gray stitch', 'selvet', 'stain',
        'chemical hazard', 'fire', 'no helmet', 'smoke', 'water leak'
    ]
    
    # Collect sample images for each class
    class_sample_imgs = {cls_name: [] for cls_name in ALL_CLASSES_LIST}
    val_lbl_dir = COMBINED_DATASET / 'labels' / 'val'
    
    SUPPORTED_EXT = {'.jpg', '.jpeg', '.png'}
    val_imgs = [p for p in val_img_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXT]
    
    for img_path in val_imgs:
        lbl_path = val_lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            continue
        with open(lbl_path) as f:
            lines = [l.strip() for l in f if l.strip()]
        for line in lines:
            cls_id = int(line.split()[0])
            cls_name = ALL_CLASSES_LIST[cls_id] if cls_id < len(ALL_CLASSES_LIST) else '?'
            if len(class_sample_imgs[cls_name]) < 2:
                class_sample_imgs[cls_name].append(img_path)
    
    # Priority classes for display
    display_groups = {
        'Strong Textile (contamination, stain)': ['contamination', 'stain'],
        'Weak Textile (color issues, selvet, gray stitch)': ['color issues', 'selvet', 'gray stitch'],
        'New Hazard: Fire & Smoke': ['fire', 'smoke'],
        'New Hazards': ['chemical hazard', 'fire', 'no helmet', 'smoke', 'water leak'],
    }
    
    INFERENCE_OUTPUT_DIR = REPORTS_DIR / 'qualitative_examples'
    INFERENCE_OUTPUT_DIR.mkdir(exist_ok=True)
    
    for group_title, cls_list in display_groups.items():
        group_imgs = []
        for cls_name in cls_list:
            group_imgs += class_sample_imgs.get(cls_name, [])[:1]
        
        if not group_imgs:
            print(f'No sample images found for: {group_title}')
            continue
        
        fig, axes = plt.subplots(1, len(group_imgs), figsize=(5 * len(group_imgs), 5))
        if len(group_imgs) == 1:
            axes = [axes]
        
        for ax, img_path in zip(axes, group_imgs):
            results = expanded_model.predict(
                source=str(img_path), verbose=False, conf=0.25
            )
            annotated = results[0].plot()  # Returns numpy array (BGR)
            # Convert BGR → RGB
            import cv2
            annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
            ax.imshow(annotated_rgb)
            ax.set_title(img_path.name[:25], fontsize=8)
            ax.axis('off')
        
        fig.suptitle(group_title, fontsize=11, fontweight='bold')
        plt.tight_layout()
        safe_name = group_title.replace(' ', '_').replace('(', '').replace(')', '').replace(',', '').replace(':', '')
        save_path = INFERENCE_OUTPUT_DIR / f'{safe_name[:50]}.png'
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        plt.close(fig)
        show_saved_png(save_path, width=950)
        print(f'Saved: {save_path}')


---


In [ ]:
import datetime

def fmt(v, pct=True):
    if v is None: return 'PENDING'
    return f'{v*100:.2f}%' if pct else str(v)

torch_available = torch is not None
cuda_available = bool(torch_available and torch.cuda.is_available())
ultralytics_version = ultralytics.__version__ if ultralytics is not None else 'not installed'

lines = [
    '# Vision-Driven Industrial Safety & Quality Inspection Engine',
    '## Final Report - Task 5',
    '',
    f'**Generated:** {datetime.datetime.now().isoformat()}',
    '**Organization:** Devlogix Technology',
    '',
    '---',
    '',
    '## Environment',
    '',
    f'- GPU: {torch.cuda.get_device_name(0) if cuda_available else "N/A"}',
    f'- VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB' if cuda_available else '- VRAM: N/A',
    f'- PyTorch: {torch.__version__ if torch_available else "not installed"}',
    f'- Ultralytics: {ultralytics_version}',
    '',
    '---',
    '',
    '## Dataset',
    '',
    '### Textile Dataset (existing)',
    '- Source: FabricDefectNTU (Kaggle: muhammadharisabid/fabricdefectntu)',
    '- Classes (7): baekra, color issues, contamination, cut, gray stitch, selvet, stain',
    '- Images: 4,742 (train: 3,900 / val: 562 / test: 280)',
    '',
    '### New Hazard Datasets',
    '- Source: Industrial Hazards Detection (Kaggle: vigneshnachu/industrial-hazards-detection)',
    '- Classes (5): chemical hazard, fire, no helmet, smoke, water leak',
    '- These classes are appended after the 7 textile classes in the combined dataset.',
    '',
    '---',
    '',
    '## Historical Baseline - Experiment A',
    '',
    f'| Metric | Value |',
    '|---|---|',
    f'| Precision | {fmt(EXPERIMENT_A["overall"]["precision"])} |',
    f'| Recall | {fmt(EXPERIMENT_A["overall"]["recall"])} |',
    f'| mAP@50 | {fmt(EXPERIMENT_A["overall"]["mAP50"])} |',
    f'| mAP@50-95 | {fmt(EXPERIMENT_A["overall"]["mAP50_95"])} |',
    '',
    '---',
    '',
    '## Improved 7-Class Model - Experiment B',
    '',
]  

if EXPERIMENT_B:
    b = EXPERIMENT_B['overall']
    lines += [
        f'| Metric | Value |',
        '|---|---|',
        f'| Precision | {fmt(b["precision"])} |',
        f'| Recall | {fmt(b["recall"])} |',
        f'| mAP@50 | {fmt(b["mAP50"])} |',
        f'| mAP@50-95 | {fmt(b["mAP50_95"])} |',
        '',
        '### Per-class Experiment B',
        '',
        '| Class | Hist mAP@50 | Improved mAP@50 | Δ |',
        '|---|---|---|---|',
    ]
    for cls_name in ['baekra','color issues','contamination','cut','gray stitch','selvet','stain']:
        hist = EXPERIMENT_A['per_class_map50'].get(cls_name, 0)
        imp = EXPERIMENT_B['per_class'].get(cls_name, {}).get('ap50')
        delta = f'{(imp - hist)*100:+.1f}%' if imp else 'N/A'
        lines.append(f'| {cls_name} | {hist*100:.1f}% | {fmt(imp) if imp else "N/A"} | {delta} |')
else:
    lines.append('Results pending - run Notebook 03.')

lines += [
    '',
    '---',
    '',
    '## Expanded 12-Class Model - Experiment C',
    '',
]

if EXPERIMENT_C:
    c_ov = EXPERIMENT_C['overall']
    lines += [
        f'| Metric | Value |',
        '|---|---|',
        f'| Precision | {fmt(c_ov["precision"])} |',
        f'| Recall | {fmt(c_ov["recall"])} |',
        f'| mAP@50 | {fmt(c_ov["mAP50"])} |',
        f'| mAP@50-95 | {fmt(c_ov["mAP50_95"])} |',
        '',
        '### All 12 Classes - Experiment C',
        '',
        '| Class | P% | R% | AP@50% | AP@50-95% |',
        '|---|---|---|---|---|',
    ]
    for cls_name in ALL_CLASSES_LIST:
        m = EXPERIMENT_C['per_class'].get(cls_name, {})
        lines.append(f'| {cls_name} | {m.get("precision",0)*100:.1f} | {m.get("recall",0)*100:.1f} | {m.get("ap50",0)*100:.1f} | {m.get("ap50_95",0)*100:.1f} |')
else:
    lines.append('Results pending - run Notebook 04.')

lines += [
    '',
    '---',
    '',
    '## Definition of Done',
    '',
    '**Criterion:** At least 2 of 5 hazard classes achieve >= 70% mAP@50, plus at least 1 of the remaining 3 achieves >= 60% mAP@50',
    '',
]

if EXPERIMENT_C:
    hazard_classes = ['chemical hazard', 'fire', 'no helmet', 'smoke', 'water leak']
    high_pass = []
    support_pass = []
    for h in hazard_classes:
        ap50 = EXPERIMENT_C['per_class'].get(h, {}).get('ap50', 0)
        if ap50 >= 0.70:
            high_pass.append(h)
        elif ap50 >= 0.60:
            support_pass.append(h)
    remaining_count = len(hazard_classes) - len(high_pass)
    for h in ['chemical hazard', 'fire', 'no helmet', 'smoke', 'water leak']:
        ap50 = EXPERIMENT_C['per_class'].get(h, {}).get('ap50', 0)
        if ap50 >= 0.70:
            status = 'HIGH PASS'
        elif ap50 >= 0.60:
            status = 'SUPPORT PASS'
        else:
            status = 'FAIL'
        lines.append(f'- {h}: {ap50*100:.1f}% {status}')
    dod_verdict = 'PASS' if len(high_pass) >= 2 and len(support_pass) >= 1 else 'FAIL'
    lines.append(f'\n**DoD Result: {dod_verdict} ({len(high_pass)}/5 >=70%, {len(support_pass)}/{remaining_count} remaining >=60%)**')
else:
    lines.append('DoD verdict pending - run Notebook 04.')

lines += [
    '',
    '---',
    '',
    '## Industrial Safety Note',
    '',
    'The model detects chemical hazard, fire, no helmet, smoke, and water leak.',
    'Proximity hazard is determined by a configurable post-processing rule:',
    '- Compute normalized center-to-center distance between forklift and person bounding boxes',
    '- Threshold DANGER: < 20% of image diagonal',
    '- Threshold WARNING: < 35% of image diagonal',
    '- This is a rule-based spatial calculation, NOT a model-predicted output.',
    '- Safety-critical deployment would require additional validation beyond this POC.',
    '',
    '---',
    '',
    '## Confusion Matrix Findings',
    '',
]

if 'CONFUSION_FINDINGS' in globals():
    for finding in CONFUSION_FINDINGS.get('hazard_summary', []):
        lines.append(f'- {finding}')
    for finding in CONFUSION_FINDINGS.get('textile_summary', []):
        lines.append(f'- {finding}')
    if CONFUSION_FINDINGS.get('requires_visual_review'):
        lines += ['', '### Visual Review Priorities', '']
        for item in CONFUSION_FINDINGS['requires_visual_review']:
            lines.append(f'- {item}')
else:
    lines.append('Confusion analysis pending - run the analysis cell before generating the report.')

lines += [
    '',
    '---',
    '',
    '*Report generated by Notebook 04 - Vision-Driven Industrial Safety & Quality Inspection Engine*',
]

report_md = '\n'.join(lines)
report_path = REPORTS_DIR / 'final_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_md)
print(f'Final report saved: {report_path}')

# Backup to Drive
if DRIVE_PROJECT:
    drive_reports = DRIVE_PROJECT / 'Training_Results' / 'hazard_reports'
    drive_reports.mkdir(parents=True, exist_ok=True)
    shutil.copy(report_path, drive_reports / 'final_report.md')
    # Copy all report images
    for f in REPORTS_DIR.glob('*.png'):
        shutil.copy(f, drive_reports / f.name)
    for f in (REPORTS_DIR / 'qualitative_examples').glob('*.png'):
        shutil.copy(f, drive_reports / f.name)
    print(f'All reports backed up to: {drive_reports}')

print('\n✅ Evaluation complete. All outputs saved.')


---
## End of Notebook 04

**All notebooks complete.** 

**Files expected on Google Drive under `Training_Results/`:**
- `Training_Results/yolov8s_improved_7class/best.pt` - Experiment B checkpoint
- `Training_Results/yolov8s_improved_7class/results.json` - Experiment B metrics
- `Training_Results/Expanded_12-class_model/best.pt` - Experiment C checkpoint
- `Training_Results/Expanded_12-class_model/expanded_12class_results.json` - Experiment C metrics
- `Training_Results/hazard_reports/final_report.md`
- `Training_Results/hazard_reports/exp_b_confusion_matrix.png`
- `Training_Results/hazard_reports/exp_c_confusion_matrix.png`
- `Training_Results/hazard_reports/qualitative_examples/*.png`
- `Training_Results/hazard_reports/combined_dataset_statistics.json`

**Next steps:**
1. Confirm `Backend/best.pt` contains the expanded model checkpoint
2. The FastAPI backend and frontend are class-agnostic - no code changes required
